# RAG Live Project — Complete Workshop Notebook

This notebook walks through building a complete Retrieval-Augmented Generation (RAG) system step by step.

**What we're building:** A policy assistant for NovaTech Solutions that answers employee questions using actual company documents.

**Steps:**
1. Load & Inspect Documents
2. Chunking
3. Embed & Store in Vector Database
4. Complete RAG Chain
5. RAG vs No-RAG Comparison
6. Production-Ready RAG
7. Advanced Patterns

---
## Setup: Install & Import Dependencies

In [ ]:
# Uncomment and run if packages are not installed
# !pip install langchain langchain-community langchain-openai langchain-chroma
# !pip install chromadb sentence-transformers python-dotenv docx2txt python-docx
# !pip install tiktoken numpy

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env file!"
print("API key loaded successfully!")

---
# Step 1: Load & Inspect Documents

**WHAT WE'RE DOING:** Loading our company's policy documents and understanding what we're working with before any RAG magic.

This is the "look at your data first" step — always do this.

### 1A: Load documents from our sample_docs folder

In [ ]:
from docx import Document

docs_folder = Path("sample_docs")

print("=" * 60)
print("STEP 1: Loading and Inspecting Documents")
print("=" * 60)

# List all files in our docs folder
print(f"\nDocuments found in '{docs_folder}':")
print("-" * 40)

documents = {}
for filepath in sorted(docs_folder.glob("*.docx")):
    doc = Document(filepath)
    content = "\n".join(para.text for para in doc.paragraphs)
    documents[filepath.name] = content
    print(f"  {filepath.name:30} -{len(content):,} characters, ~{len(content.split()):,} words")

print(f"\nTotal documents: {len(documents)}")
total_chars = sum(len(c) for c in documents.values())
total_words = sum(len(c.split()) for c in documents.values())
print(f"Total content: {total_chars:,} characters, ~{total_words:,} words")

### 1B: Look at one document to understand the content

In [ ]:
preview_file = "03_Leave_Attendance_Policy.docx"
print(f"PREVIEW: First 500 characters of '{preview_file}'")
print("=" * 60)
print(documents[preview_file][:500])
print("...")

### 1C: Estimate token count (for cost awareness)

In [ ]:
try:
    import tiktoken
    enc = tiktoken.encoding_for_model("gpt-4o-mini")
    total_tokens = sum(len(enc.encode(c)) for c in documents.values())
    print(f"TOKEN ESTIMATE (gpt-4o-mini tokenizer)")
    print(f"{'=' * 60}")
    print(f"Total tokens across all documents: {total_tokens:,}")
    print(f"Estimated embedding cost: negligible (we use a free local model)")
    print(f"Estimated LLM cost per query: ~$0.0001 (gpt-4o-mini)")
except ImportError:
    print("(tiktoken not installed — skipping token count)")

### 1D: Why we can't just dump everything into one prompt

In [ ]:
print(f"WHY WE NEED RAG (not just a big prompt)")
print(f"{'=' * 60}")
print(f"Total content: ~{total_words:,} words ~ ~{int(total_words / 0.75):,} tokens")
print(f"GPT-4o-mini context window: 128,000 tokens")
print(f"Our documents fit in one prompt... but in production:")
print(f"  ->500 documents would be ~{total_words * 100:,} tokens — way too large")
print(f"  ->Most content is irrelevant to any specific question")
print(f"  ->Cost scales linearly with input tokens")
print(f"  ->RAG retrieves ONLY the relevant chunks — efficient & accurate")

---
# Step 2: Chunking

**WHAT WE'RE DOING:** Splitting documents into smaller pieces (chunks) that can be individually retrieved. This is the most underrated step in RAG — get it wrong and nothing downstream works.

**KEY CONCEPT:** `chunk_size` and `chunk_overlap` control the trade-off between context preservation and retrieval precision.

### 2A: Load documents using LangChain's loader

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("=" * 60)
print("STEP 2: Chunking Documents")
print("=" * 60)

# DirectoryLoader loads all matching files from a folder
loader = DirectoryLoader(
    "sample_docs",
    glob="*.docx",
    loader_cls=Docx2txtLoader,
)
raw_documents = loader.load()

print(f"\nLoaded {len(raw_documents)} documents via LangChain")
for doc in raw_documents:
    source = Path(doc.metadata["source"]).name
    print(f"  {source:30} — {len(doc.page_content):,} chars")

### 2B: Chunk with RecursiveCharacterTextSplitter

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,        # Target size in characters
    chunk_overlap=50,      # Characters of overlap between adjacent chunks
    length_function=len,   # How to measure length (characters here)
    separators=["\n\n", "\n", ". ", " ", ""],  # Try these in order
)

chunks = splitter.split_documents(raw_documents)

print(f"Original documents: {len(raw_documents)}")
print(f"After chunking:     {len(chunks)} chunks")
print(f"Chunk size target:  500 characters")
print(f"Overlap:            50 characters")

### 2C: Inspect the chunks

In [ ]:
# Show the first 5 chunks
for i, chunk in enumerate(chunks[:5]):
    source = Path(chunk.metadata["source"]).name
    print(f"\n--- Chunk {i} (from {source}) ---")
    print(f"Length: {len(chunk.page_content)} chars")
    print(f"Content preview: {chunk.page_content[:200]}...")

# Show chunk size distribution
sizes = [len(c.page_content) for c in chunks]
print(f"\n{'=' * 60}")
print("CHUNK SIZE DISTRIBUTION")
print(f"{'=' * 60}")
print(f"  Min:     {min(sizes)} chars")
print(f"  Max:     {max(sizes)} chars")
print(f"  Average: {sum(sizes) / len(sizes):.0f} chars")
print(f"  Median:  {sorted(sizes)[len(sizes)//2]} chars")

### 2D: Demonstrate overlap

In [ ]:
# Find two adjacent chunks from the same document
for i in range(len(chunks) - 1):
    if chunks[i].metadata["source"] == chunks[i + 1].metadata["source"]:
        end_of_chunk_i = chunks[i].page_content[-80:]
        start_of_chunk_j = chunks[i + 1].page_content[:80]
        print(f"End of chunk {i}:")
        print(f'  "...{end_of_chunk_i}"')
        print(f"\nStart of chunk {i + 1}:")
        print(f'  "{start_of_chunk_j}..."')

        # Find the overlap
        overlap_text = ""
        for length in range(min(80, len(end_of_chunk_i), len(start_of_chunk_j)), 0, -1):
            if end_of_chunk_i.endswith(start_of_chunk_j[:length]):
                overlap_text = start_of_chunk_j[:length]
                break

        if overlap_text:
            print(f"\n  OVERLAP ({len(overlap_text)} chars):")
            print(f'  "{overlap_text}"')
        else:
            print(f"\n  (Overlap exists but split at a separator boundary)")
        print(f"\n  ^ This overlap ensures no information is lost at chunk boundaries")
        break

### 2E: Compare different chunk sizes

In [ ]:
print("EXPERIMENT: Different Chunk Sizes")
print("=" * 60)

for size in [200, 500, 1000, 2000]:
    test_splitter = RecursiveCharacterTextSplitter(
        chunk_size=size, chunk_overlap=50
    )
    test_chunks = test_splitter.split_documents(raw_documents)
    avg_size = sum(len(c.page_content) for c in test_chunks) / len(test_chunks)
    print(f"  chunk_size={size:5}  ->  {len(test_chunks):3} chunks  (avg {avg_size:.0f} chars each)")

print(f"\n  -> Smaller chunks = more pieces, finer retrieval, but may lose context")
print(f"  -> Larger chunks = fewer pieces, more context per chunk, but less precise")
print(f"  -> We'll use 500 as our working default")

### Metadata check

In [ ]:
sample = chunks[0]
print(f"Each chunk carries metadata: {sample.metadata}")
print(f"This lets us trace answers back to source documents!")

---
# Step 3: Embed & Store in Vector Database

**WHAT WE'RE DOING:** Taking our text chunks and converting them into vectors using an embedding model, then storing them in ChromaDB for fast similarity search.

**KEY CONCEPT:** The embedding model (`all-MiniLM-L6-v2`) and the LLM (`gpt-4o-mini`) are **TWO DIFFERENT MODELS** doing **TWO DIFFERENT JOBS**.

### 3A: Recreate our chunks (from Step 2)

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import time
import numpy as np

# Re-chunk (in case you jumped here directly)
loader = DirectoryLoader(
    "sample_docs", glob="*.docx",
    loader_cls=Docx2txtLoader,
)
raw_docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)
print(f"{len(chunks)} chunks ready for embedding")

### 3B: Load the embedding model

In [ ]:
# This is the EMBEDDING model — small, free, runs on CPU
# It converts text -> vector (384 dimensions)
# NOT the same as the LLM that generates answers!
embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
)

print("Model: all-MiniLM-L6-v2")
print("Type:  Sentence embedding model (NOT an LLM)")
print("Size:  ~80MB, runs on CPU")
print("Output: 384-dimensional vectors")

### 3C: See what an embedding looks like

In [ ]:
sample_text = "What is the SLA for containing a critical cybersecurity incident?"
sample_vector = embedding_model.embed_query(sample_text)

print(f"Input text: \"{sample_text}\"")
print(f"Output vector: {len(sample_vector)} dimensions")
print(f"First 10 values: {[round(v, 4) for v in sample_vector[:10]]}")
print(f"Last 10 values:  {[round(v, 4) for v in sample_vector[-10:]]}")

### Cosine Similarity Demo — similar texts have similar vectors

In [ ]:
text_a = "What is the laptop refresh cycle?"
text_b = "How often are company laptops replaced?"
text_c = "What is the maternity leave entitlement?"

vec_a = embedding_model.embed_query(text_a)
vec_b = embedding_model.embed_query(text_b)
vec_c = embedding_model.embed_query(text_c)

def cosine_sim(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

print(f"COSINE SIMILARITY DEMO")
print(f"{'=' * 60}")
print(f"\n  A: \"{text_a}\"")
print(f"  B: \"{text_b}\"")
print(f"  C: \"{text_c}\"")
print(f"\n  sim(A, B) = {cosine_sim(vec_a, vec_b):.4f}  <- Similar meaning!")
print(f"  sim(A, C) = {cosine_sim(vec_a, vec_c):.4f}  <- Different topics")
print(f"  sim(B, C) = {cosine_sim(vec_b, vec_c):.4f}  <- Different topics")
print(f"\n  -> The embedding model captures that A and B are about the same thing,")
print(f"     even though they use completely different words.")

### 3D: Create the vector store (ChromaDB)

In [ ]:
PERSIST_DIR = "./chroma_db"

start_time = time.time()

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=PERSIST_DIR,
    collection_name="technova_policies",
)

elapsed = time.time() - start_time
print(f"Embedded and stored {len(chunks)} chunks in {elapsed:.1f} seconds")
print(f"Persist directory: {PERSIST_DIR}")
print(f"Collection name: 'technova_policies'")
print(f"Each chunk is now a 384-dim vector in the database")

### 3E: Test retrieval (WITHOUT LLM — just the vector DB)

In [ ]:
print("TESTING RETRIEVAL (no LLM yet — just vector search)")
print("=" * 60)

test_questions = [
    "How many sick days do I get per year?",
    "What is the minimum password length for NovaTech accounts?",
    "What is the daily hotel limit for Tier 1 city business travel?",
    "How do I report a cybersecurity incident?",
    "What rating do I need for a promotion?",
]

for question in test_questions:
    print(f"\n  Q: \"{question}\"")
    results = vectorstore.similarity_search_with_score(question, k=2)
    for i, (doc, score) in enumerate(results):
        source = Path(doc.metadata["source"]).name
        preview = doc.page_content[:100].replace("\n", " ")
        print(f"    [{i+1}] score={score:.4f} | {source}")
        print(f'        "{preview}..."')

print(f"\nKEY OBSERVATION:")
print("The vector DB retrieves the RIGHT chunks for each question —")
print("leave questions get leave chunks, security questions get security chunks.")
print("This works even though the question wording differs from the document text!")

---
# Step 4: Complete RAG Chain

**WHAT WE'RE DOING:** Connecting the retriever to an LLM to build the complete RAG pipeline. This is where everything comes together.

**KEY CONCEPT:** We first build the prompt MANUALLY so you see exactly what the LLM receives. Then we wire it up with LangChain.

### 4A: Load vector store & LLM

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.messages import HumanMessage

PERSIST_DIR = "./chroma_db"

# Try to load existing DB, or create fresh
if Path(PERSIST_DIR).exists():
    print("Loading existing vector store...")
    vectorstore = Chroma(
        persist_directory=PERSIST_DIR,
        embedding_function=embedding_model,
        collection_name="technova_policies",
    )
    print(f"  Loaded. Collection has {vectorstore._collection.count()} chunks.")
else:
    print("Creating vector store from scratch...")
    loader = DirectoryLoader(
        "sample_docs", glob="*.docx",
        loader_cls=Docx2txtLoader,
    )
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = splitter.split_documents(loader.load())
    vectorstore = Chroma.from_documents(
        documents=chunks, embedding=embedding_model,
        persist_directory=PERSIST_DIR, collection_name="technova_policies",
    )
    print(f"  Created with {len(chunks)} chunks.")

# Create retriever (top 3 most similar chunks)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)

### 4B: The Manual Approach (see exactly what the LLM gets)

In [ ]:
question = "What is the minimum password length and expiry policy for NovaTech accounts?"

# Step 1: Retrieve relevant chunks
retrieved_docs = retriever.invoke(question)

print(f"Question: \"{question}\"")
print(f"\nRetrieved {len(retrieved_docs)} chunks:")
for i, doc in enumerate(retrieved_docs):
    source = Path(doc.metadata["source"]).name
    print(f"\n  --- Chunk {i+1} (from {source}) ---")
    print(f"  {doc.page_content[:200]}...")

In [ ]:
# Step 2: Build the augmented prompt manually
context = "\n\n".join([doc.page_content for doc in retrieved_docs])

manual_prompt = f"""You are a helpful policy assistant for NovaTech Solutions Pvt. Ltd.
Answer the employee's question based ONLY on the provided context.
If the context doesn't contain the answer, say "I don't have that information in the available documents."

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""

print("THE ACTUAL PROMPT SENT TO THE LLM")
print("(This is the 'augmented' in Retrieval-Augmented Generation)")
print("=" * 60)
print(f"\n{manual_prompt[:1000]}")
if len(manual_prompt) > 1000:
    print(f"... [{len(manual_prompt) - 1000} more characters] ...")
print(f"\nTotal prompt length: {len(manual_prompt)} characters")

In [ ]:
# Step 3: Send to LLM
print("LLM RESPONSE (manual approach)")
print("=" * 60)

response = llm.invoke([HumanMessage(content=manual_prompt)])
print(f"\n{response.content}")

### 4C: The LangChain Approach (same thing, cleaner code)

In [ ]:
# Define the prompt template
template = """You are a helpful policy assistant for NovaTech Solutions Pvt. Ltd.
Answer the employee's question based ONLY on the provided context.
If the context doesn't contain the answer, say "I don't have that information in the available documents."
Keep your answer concise and specific.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""

prompt = ChatPromptTemplate.from_template(template)

# Helper: format retrieved docs into a single string
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Build the chain: retriever -> format -> prompt -> LLM -> parse output
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

# Test it!
answer = rag_chain.invoke(question)
print(f"Q: {question}")
print(f"A: {answer}")

### 4D: Test with multiple questions

In [ ]:
test_questions = [
    "How many days of earned leave do employees get per year?",
    "What is the SLA for a P1 critical cybersecurity incident?",
    "What happens if I get a rating of 2 in my performance review?",
    "What is the daily hotel limit for international travel to North America?",
    "What is the laptop refresh cycle at NovaTech?",
    "How do I report a security incident and what is the hotline number?",
    "What are the core working hours for remote employees?",
]

for q in test_questions:
    answer = rag_chain.invoke(q)
    print(f"\nQ: {q}")
    print(f"A: {answer}")
    print("-" * 40)

### 4E: RAG with source citations

In [ ]:
def rag_with_sources(question):
    """RAG that returns both the answer and the source chunks."""
    docs = retriever.invoke(question)
    context = format_docs(docs)

    chain = prompt | llm | StrOutputParser()
    answer = chain.invoke({"context": context, "question": question})

    sources = set()
    for doc in docs:
        sources.add(Path(doc.metadata["source"]).name)

    return {
        "question": question,
        "answer": answer,
        "sources": list(sources),
        "num_chunks_used": len(docs),
    }

# Demo with sources
result = rag_with_sources("What happens if I lose my company laptop and how soon must I report it?")
print(f"Q: {result['question']}")
print(f"A: {result['answer']}")
print(f"Sources: {', '.join(result['sources'])}")
print(f"Chunks used: {result['num_chunks_used']}")

---
# Step 5: RAG vs. No-RAG — The Hallucination Demo

**WHAT WE'RE DOING:** Asking the SAME questions WITH and WITHOUT RAG to demonstrate exactly why RAG matters.

**KEY INSIGHT:** Without RAG, the LLM hallucinates confident-sounding but completely fabricated company policies.

### Setup: RAG chain vs No-RAG chain

In [ ]:
# RAG chain (reuse from above)
rag_template = """You are a helpful policy assistant for NovaTech Solutions Pvt. Ltd.
Answer the employee's question based ONLY on the provided context.
If the context doesn't contain the answer, say "I don't have that information."

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""

rag_prompt = ChatPromptTemplate.from_template(rag_template)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt | llm | StrOutputParser()
)

# No-RAG chain (LLM answers from its own knowledge)
no_rag_template = """You are a helpful policy assistant for NovaTech Solutions Pvt. Ltd.
Answer the employee's question about company policies.

QUESTION: {question}

ANSWER:"""

no_rag_prompt = ChatPromptTemplate.from_template(no_rag_template)
no_rag_chain = (
    {"question": RunnablePassthrough()}
    | no_rag_prompt | llm | StrOutputParser()
)

print("Both chains ready!")

### 5A: Side-by-side comparison

In [ ]:
questions = [
    "How many days of earned leave do employees get per year at NovaTech?",
    "What is the minimum password length and how often must it be changed?",
    "What is the SLA for containing a P1 critical cybersecurity incident?",
    "What is the daily hotel limit for business travel to North America?",
    "What is the employee referral bonus for a technical role at Band 3 or above?",
]

for i, question in enumerate(questions, 1):
    print(f"\n{'=' * 60}")
    print(f"QUESTION {i}: {question}")
    print(f"{'=' * 60}")

    # Without RAG
    no_rag_answer = no_rag_chain.invoke(question)
    print(f"\n[X] WITHOUT RAG (LLM guesses):")
    print(f"   {no_rag_answer}")

    # With RAG
    rag_answer = rag_chain.invoke(question)
    print(f"\n[OK] WITH RAG (LLM uses retrieved documents):")
    print(f"   {rag_answer}")

    # Show what was retrieved
    docs = retriever.invoke(question)
    sources = set(Path(d.metadata["source"]).name for d in docs)
    print(f"\n   Sources used: {', '.join(sources)}")

### 5B: Killer Demo — Highly specific factual question

In [ ]:
specific_q = "If I want to work remotely from another country, how many days am I allowed per year and whose approval do I need?"

print(f"Q: {specific_q}")

no_rag_ans = no_rag_chain.invoke(specific_q)
print(f"\n[X] WITHOUT RAG:")
print(f"   {no_rag_ans}")

rag_ans = rag_chain.invoke(specific_q)
print(f"\n[OK] WITH RAG:")
print(f"   {rag_ans}")

print(f"\nGROUND TRUTH (from 06_Remote_Work_Policy.docx):")
print(f"   45 days per year, requires CHRO + Legal + Finance approval")

### 5C: Edge Case — Question not in the documents

In [ ]:
unknown_q = "What is NovaTech's policy on cryptocurrency reimbursement for employee expenses?"

print(f"Q: {unknown_q}")

no_rag_ans = no_rag_chain.invoke(unknown_q)
print(f"\n[X] WITHOUT RAG:")
print(f"   {no_rag_ans}")

rag_ans = rag_chain.invoke(unknown_q)
print(f"\n[OK] WITH RAG:")
print(f"   {rag_ans}")

print(f"\n-> The RAG system correctly says it doesn't have that information,")
print(f"   while the LLM without RAG invents a plausible-sounding policy!")

### Key Takeaways

1. **WITHOUT RAG:** The LLM confidently invents policies that sound real but are completely fabricated. This is hallucination.
2. **WITH RAG:** The LLM answers from actual documents and gets the specific numbers and details correct.
3. **WHEN INFO IS MISSING:** RAG systems correctly decline to answer, while bare LLMs fabricate an answer.
4. **THE FIX IS IN THE INPUT:** Same LLM, same temperature, same everything — the only difference is the retrieved context. RAG doesn't fix the model. It fixes the input.

---
# Step 6: Making RAG Production-Ready

**WHAT WE'RE DOING:** The basic RAG works. Now we learn what breaks in production and how to fix it.

**TOPICS:** Chunking experiments, retrieval debugging, failure modes, metadata filtering, evaluation basics.

### 6A: How chunk size affects answer quality

In [ ]:
print("6A: HOW CHUNK SIZE AFFECTS ANSWER QUALITY")
print("=" * 60)

question = "What are the increment percentages for each performance rating level?"

rag_template_6 = """Answer based ONLY on the context. Be specific with numbers.
If the context doesn't contain the answer, say "Not found in context."

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""

prompt_6 = ChatPromptTemplate.from_template(rag_template_6)

for chunk_size in [200, 500, 1000]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=50
    )
    exp_chunks = splitter.split_documents(raw_docs)

    # Create temporary vectorstore for this experiment
    temp_vs = Chroma.from_documents(
        documents=exp_chunks, embedding=embedding_model,
        collection_name=f"experiment_{chunk_size}",
    )
    temp_retriever = temp_vs.as_retriever(search_kwargs={"k": 3})

    # Get retrieved chunks
    docs = temp_retriever.invoke(question)
    context = "\n\n".join(d.page_content for d in docs)

    # Get answer
    chain = prompt_6 | llm | StrOutputParser()
    answer = chain.invoke({"context": context, "question": question})

    print(f"\n  chunk_size={chunk_size}:")
    print(f"  Retrieved {len(docs)} chunks, total context: {len(context)} chars")
    print(f"  Answer: {answer[:200]}...")

    # Cleanup
    temp_vs.delete_collection()

print(f"\n  -> Notice: too-small chunks may miss the increment table entirely")
print(f"  -> Too-large chunks include irrelevant content that can confuse the LLM")

### 6B: Debugging — Retrieval vs. Generation Problems

In [ ]:
print("6B: DEBUGGING — Retrieval vs. Generation Problems")
print("=" * 60)

vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embedding_model,
    collection_name="technova_policies",
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def debug_rag(question):
    """Debug function: shows what was retrieved and the final answer."""
    print(f"\n  Q: \"{question}\"")

    # Step 1: Check what was retrieved
    docs = retriever.invoke(question)
    print(f"\n  RETRIEVED CHUNKS:")
    for i, doc in enumerate(docs):
        source = Path(doc.metadata["source"]).name
        print(f"    [{i+1}] {source}: \"{doc.page_content[:100]}...\"")

    # Step 2: Check if the answer is IN the retrieved chunks
    context = "\n\n".join(d.page_content for d in docs)

    chain = prompt_6 | llm | StrOutputParser()
    answer = chain.invoke({"context": context, "question": question})
    print(f"\n  ANSWER: {answer}")

    # Diagnosis
    print(f"\n  DIAGNOSIS:")
    sources = set(Path(d.metadata["source"]).name for d in docs)
    print(f"    Sources used: {', '.join(sources)}")
    print(f"    -> If wrong source: RETRIEVAL problem (fix embeddings/chunks)")
    print(f"    -> If right source but wrong answer: GENERATION problem (fix prompt)")

# Test with tricky questions
debug_rag("What is the notice period for a Band 5 Senior Manager?")
print()
debug_rag("What security tools does NovaTech use for endpoint protection?")

### 6C: Metadata Filtering — Narrow retrieval to specific docs

In [ ]:
print("6C: METADATA FILTERING")
print("=" * 60)

# Rebuild with richer metadata
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

enriched_chunks = []
for doc in raw_docs:
    source_name = Path(doc.metadata["source"]).name
    category_map = {
        "01_Employee_Handbook_Code_of_Conduct.docx": "Compliance",
        "02_Recruitment_Onboarding_Policy.docx": "HR",
        "03_Leave_Attendance_Policy.docx": "HR",
        "04_Performance_Management_Policy.docx": "HR",
        "05_Compensation_Benefits_Policy.docx": "Finance",
        "06_Remote_Work_Policy.docx": "HR",
        "07_Grievance_Disciplinary_Policy.docx": "HR",
        "08_POSH_Diversity_Inclusion_Policy.docx": "Compliance",
        "09_Separation_Offboarding_Policy.docx": "HR",
        "10_IT_Acceptable_Use_Policy.docx": "IT",
        "11_Data_Security_Privacy_Policy.docx": "IT",
        "12_IT_Asset_Management_Policy.docx": "IT",
        "13_Cybersecurity_Incident_Response_Policy.docx": "IT",
        "14_Learning_Development_Policy.docx": "HR",
        "15_Travel_Expense_Policy.docx": "Finance",
    }
    category = category_map.get(source_name, "General")

    doc_chunks = splitter.split_documents([doc])
    for chunk in doc_chunks:
        chunk.metadata["category"] = category
        chunk.metadata["document"] = source_name
    enriched_chunks.extend(doc_chunks)

# Create new vectorstore with metadata
meta_vs = Chroma.from_documents(
    documents=enriched_chunks,
    embedding=embedding_model,
    collection_name="technova_with_metadata",
)

# Search with metadata filter
print("\nSearch WITHOUT filter (all documents):")
results = meta_vs.similarity_search("data encryption and password policy", k=3)
for r in results:
    print(f"  [{r.metadata['category']}] {r.metadata['document']}: {r.page_content[:80]}...")

print("\nSearch WITH filter (IT category only):")
results = meta_vs.similarity_search(
    "data encryption and password policy",
    k=3,
    filter={"category": "IT"},
)
for r in results:
    print(f"  [{r.metadata['category']}] {r.metadata['document']}: {r.page_content[:80]}...")

print("\n  -> Metadata filters let you narrow search to specific doc types")
print("  -> Useful for: 'search only HR docs' or 'search only docs from 2025'")

meta_vs.delete_collection()

### 6D: Common RAG Failure Modes

In [ ]:
print("COMMON RAG FAILURE MODES")
print("=" * 60)

failures = [
    ("Wrong chunks retrieved",
     "Embedding model can't distinguish domain-specific terms",
     "Use domain-specific embeddings or add metadata filters"),

    ("Right chunks but wrong answer",
     "Prompt doesn't constrain the LLM enough",
     "Improve system prompt, add 'answer ONLY from context'"),

    ("Answer spreads across multiple chunks",
     "Information was split at a chunk boundary",
     "Increase chunk size or overlap for that document type"),

    ("Stale answers after document update",
     "Vectors not re-computed after source docs changed",
     "Re-index changed documents (track doc hashes)"),

    ("Slow response time",
     "Too many chunks retrieved or context too large",
     "Reduce K, use metadata pre-filtering, compress context"),
]

for i, (mode, cause, fix) in enumerate(failures, 1):
    print(f"\n  {i}. {mode}")
    print(f"     Cause: {cause}")
    print(f"     Fix:   {fix}")

---
# Step 7: Advanced Patterns

Two advanced patterns that solve real production problems:

- **Pattern 1: Multi-Query RAG** — rephrase the question multiple ways to improve recall
- **Pattern 2: Conversational RAG** — handle follow-up questions with chat history

### 7A: Multi-Query RAG

In [ ]:
print("7A: MULTI-QUERY RAG")
print("=" * 60)
print("""
Problem: A vague question retrieves poor chunks.
Solution: Generate 3 rephrasings, retrieve for each, combine results.
This dramatically improves recall for ambiguous questions.
""")

# Reload retriever
vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embedding_model,
    collection_name="technova_policies",
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def multi_query_rag(question, k=3):
    """Generate multiple query variations and retrieve from all of them."""

    # Step 1: Generate query variations using the LLM
    rephrase_prompt = ChatPromptTemplate.from_template(
        """Generate 3 different versions of the following question.
Each version should use different words but ask about the same thing.
Return ONLY the 3 questions, one per line, no numbering.

Original question: {question}"""
    )

    chain = rephrase_prompt | llm | StrOutputParser()
    variations_text = chain.invoke({"question": question})
    variations = [q.strip() for q in variations_text.strip().split("\n") if q.strip()]

    print(f"  Original:    \"{question}\"")
    print(f"  Variations:")
    for v in variations:
        print(f"    -> \"{v}\"")

    # Step 2: Retrieve for original + all variations
    all_queries = [question] + variations
    all_docs = []
    seen_content = set()

    for query in all_queries:
        docs = retriever.invoke(query)
        for doc in docs:
            content_hash = hash(doc.page_content[:100])
            if content_hash not in seen_content:
                seen_content.add(content_hash)
                all_docs.append(doc)

    print(f"\n  Retrieved {len(all_docs)} unique chunks (from {len(all_queries)} queries)")

    # Step 3: Generate answer from all retrieved context
    context = "\n\n".join(d.page_content for d in all_docs[:6])

    answer_prompt = ChatPromptTemplate.from_template(
        """Answer based ONLY on the context. Be specific.
If the context doesn't contain the answer, say so.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""
    )

    answer_chain = answer_prompt | llm | StrOutputParser()
    answer = answer_chain.invoke({"context": context, "question": question})

    sources = set(Path(d.metadata["source"]).name for d in all_docs[:6])
    return answer, sources

In [ ]:
# Demo: vague question that benefits from multi-query
print("--- Demo: Vague question ---")
answer, sources = multi_query_rag("What should I know about data security and encryption?")
print(f"\n  Answer: {answer}")
print(f"  Sources: {', '.join(sources)}")

print("\n--- Demo: Specific question ---")
answer, sources = multi_query_rag("What are the travel expense limits and reimbursement rules?")
print(f"\n  Answer: {answer}")
print(f"  Sources: {', '.join(sources)}")

### 7B: Conversational RAG (handling follow-up questions)

In [ ]:
print("7B: CONVERSATIONAL RAG")
print("=" * 60)
print("""
Problem: "What about the second point?" — the retriever doesn't
know what "the second point" refers to without conversation history.
Solution: Rewrite follow-up questions using chat history first.
""")

# The contextualizer: rewrites follow-up questions into standalone ones
contextualize_prompt = ChatPromptTemplate.from_template(
    """Given the chat history and the latest user question, rewrite the
question to be a standalone question that doesn't need the chat history
to understand. Do NOT answer the question, just rewrite it.
If the question is already standalone, return it as-is.

CHAT HISTORY:
{chat_history}

LATEST QUESTION: {question}

STANDALONE QUESTION:"""
)

answer_prompt = ChatPromptTemplate.from_template(
    """You are a helpful policy assistant for NovaTech Solutions Pvt. Ltd.
Answer based ONLY on the provided context.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""
)


def conversational_rag(question, chat_history=None):
    """RAG that handles follow-up questions using chat history."""
    if chat_history is None:
        chat_history = []

    # Step 1: If there's history, rewrite the question
    if chat_history:
        history_text = "\n".join(
            f"{'User' if i % 2 == 0 else 'Assistant'}: {msg}"
            for i, msg in enumerate(chat_history)
        )

        rewrite_chain = contextualize_prompt | llm | StrOutputParser()
        standalone_question = rewrite_chain.invoke({
            "chat_history": history_text,
            "question": question,
        })
        print(f'  [Rewritten] "{question}" -> "{standalone_question}"')
    else:
        standalone_question = question
        print(f'  [Standalone] "{question}"')

    # Step 2: Retrieve using the standalone question
    docs = retriever.invoke(standalone_question)
    context = format_docs(docs)

    # Step 3: Generate answer
    chain = answer_prompt | llm | StrOutputParser()
    answer = chain.invoke({"context": context, "question": standalone_question})

    return answer

In [ ]:
# Simulate a multi-turn conversation
print("--- Simulating a conversation ---\n")

history = []

# Turn 1
q1 = "What is the cybersecurity incident response process at NovaTech?"
print(f"User: {q1}")
a1 = conversational_rag(q1, history)
print(f"Assistant: {a1}\n")
history.extend([q1, a1])

# Turn 2 (follow-up — needs context from turn 1)
q2 = "What about the response time SLA for critical incidents?"
print(f"User: {q2}")
a2 = conversational_rag(q2, history)
print(f"Assistant: {a2}\n")
history.extend([q2, a2])

# Turn 3 (another follow-up)
q3 = "Who leads the CSIRT team and what is the security hotline number?"
print(f"User: {q3}")
a3 = conversational_rag(q3, history)
print(f"Assistant: {a3}\n")

### Key Insight

The contextualize step is crucial:
- "What about the internet speed?" alone would retrieve random chunks.
- After rewriting: "What are the internet speed requirements for remote work at NovaTech?" retrieves the right chunks.

This is how ChatGPT-style RAG applications handle multi-turn conversations — they rewrite each question before retrieval.

---
## Experiment Time!

Try these on your own:

1. Add your own documents to `sample_docs/` and re-index
2. Try different chunk sizes (200, 1000) and compare answers
3. Ask questions that span multiple documents
4. Try to break it — find questions it gets wrong, then debug
5. Modify the system prompt and see how answers change

---

**Note:** Step 8 (Streamlit Chat Demo) is a separate file — run it with:
```bash
streamlit run step8_streamlit_demo.py
```